### Reranking

Re-ranking is a second-stage filtering process in retrieval systems, especially in RAG pipelines, where we:

1. First use a fast retriever (like BM25, FAISS, hybrid) to fetch top-k documents quickly.

2. Then use a more accurate but slower model (like a cross-encoder or LLM) to re-score and reorder those documents by relevance to the queryIt ensures that the most relevant documents appear at the top, improving the final answer from the LLM.


In [1]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.schema import Document
from langchain_core.output_parsers import StrOutputParser 


d:\project\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load text file
loader=TextLoader("data/langchain_sample.txt")
raw_docs=loader.load()

In [3]:
#Split text into document chunks
splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
docs=splitter.split_documents(raw_docs)
docs

[Document(metadata={'source': 'data/langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(metadata={'source': 'data/langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(metadata={'source': 'data/langchain_sample.txt'}, page_content='Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.'),
 Docu

In [4]:
#User query

query="How can I use langchain to build an application with memory and tools?"
#FAISS and HuggingFace Embeddings

In [5]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(docs,embedding_model)
retriever=vectorstore.as_retriever(search_kwargs={"k":8})

In [ ]:
#GROQ AI Embeddings
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"]

In [9]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

#Reranking - prompt and use LLM
llm=init_chat_model("groq:llama-3.1-8b-instant")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001D46D478AD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D46D4797F0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [10]:
#Prompt Template for reranking

prompt=PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from most to least relevant.
                                    
User Question: {question}
                                    
Documents: {documents}
                                    
Instructions:
- Think about the relevance of each document to the user's question.
- Return a list of document indices in ranked order, starting from the most relevant.
            
Output Format: comma seperated document indices (e.g., 2,1,3,0...)
""")

In [19]:
retrieved_docs=retriever.invoke(query)
retrieved_docs
chain=prompt |llm |StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template="\nYou are a helpful assistant. Your task is to rank the following documents from most to least relevant.\n\nUser Question: {question}\n\nDocuments: {documents}\n\nInstructions:\n- Think about the relevance of each document to the user's question.\n- Return a list of document indices in ranked order, starting from the most relevant.\n\nOutput Format: comma seperated document indices (e.g., 2,1,3,0...)\n")
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001D46D478AD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D46D4797F0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))
| StrOutputParser()

In [14]:
#Combine all the retrieved docs
doc_lines=[f"{i+1},{doc.page_content}" for i,doc in enumerate(retrieved_docs)]
formatted_docs="\n".join(doc_lines)


In [15]:
doc_lines

['1,LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.',
 '2,LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.',
 '3,LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.',
 '4,Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.',
 '5,Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-step t

In [16]:
formatted_docs

'1,LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.\n2,LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.\n3,LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\n4,Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.\n5,Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-step tasks like que

In [21]:
response=chain.invoke({"question":query,"documents":formatted_docs})
response

"To rank the documents, I'll assess their relevance to the user's question about using LangChain to build an application with memory and tools.\n\nBased on the question, the most relevant documents are those that discuss LangChain's capabilities, particularly its memory and tool integration features. Here's the ranking:\n\n1. 1 (LangChain is a flexible framework designed for developing applications powered by large language models (LLMs)...), \n2. 4 (Memory in LangChain enables context retention across multiple steps in a conversation or task...),\n3. 3 (LangChain supports tool integration including web search, calculators, and APIs...),\n4. 5 (Agents in LangChain are chains that use LLMs to decide which tools to use and in what order...),\n5. 6 (Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt...), \n6. 2 (LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere...),\n7. 8 

[3, 2, 4, 5, 1, 7, 6]

In [25]:
retrieved_docs

[Document(id='8200b2a7-6c95-44c3-835c-69e3e1454969', metadata={'source': 'data/langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='4bd1d11f-20d9-4669-96a5-91ae0ba46f0e', metadata={'source': 'data/langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.'),
 Document(id='2d11e79c-5366-4672-b8d6-b90ead896024', metadata={'source': 'data/langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems

In [26]:
reranked_docs=[retrieved_docs[i] for i in indices if 0<=i<len(retrieved_docs)]
reranked_docs

[Document(id='6bb4da57-dda6-467f-ad0f-f396025b4308', metadata={'source': 'data/langchain_sample.txt'}, page_content='Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='2d11e79c-5366-4672-b8d6-b90ead896024', metadata={'source': 'data/langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.'),
 Document(id='93bd876e-1819-4646-a03e-871add754866', metadata={'source': 'data/langchain_sample.txt'}, page_content='Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-step tasks like question answering with search and code execution.'),
 Document(id='8f8d361b-2abf-43dc-9998-2fefcbf2f781', metadata={'source': 'data/langchain_sample.txt'}, page_content='Retri

In [29]:
#Step 6: Output
print("\n Final Reranked results:")
for i,doc in enumerate(reranked_docs,1):
    print(f"\nRank{i}:\n{doc.page_content}")


 Final Reranked results:

Rank1:
Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.

Rank2:
LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.

Rank3:
Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-step tasks like question answering with search and code execution.

Rank4:
Retrieval-Augmented Generation (RAG) is a powerful technique where external knowledge is retrieved and passed into the prompt to ground LLM responses. LangChain makes it easy to implement RAG using vector databases like FAISS, Chroma, and Pinecone.

Rank5:
LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize 